In [ ]:
from pathlib import Path
from scipy.io import wavfile
from scipy.signal import resample_poly

def load_wav_speech(path, target_fs=8000, max_seconds=30):
    """
    Loads a .wav speech file, converts it to mono, normalizes it,
    optionally downsamples it, and limits the duration.
    """
    fs_original, x = wavfile.read(path)

    # Convert integer WAV data to float
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(float) / np.iinfo(x.dtype).max
    else:
        x = x.astype(float)

    # Stereo to mono
    if x.ndim > 1:
        x = np.mean(x, axis=1)

    # Remove DC offset
    x = x - np.mean(x)

    # Downsample if needed
    if target_fs is not None and fs_original != target_fs:
        gcd = np.gcd(fs_original, target_fs)
        up = target_fs // gcd
        down = fs_original // gcd
        x = resample_poly(x, up, down)
        fs = target_fs
    else:
        fs = fs_original

    # Limit duration
    max_samples = int(max_seconds * fs)
    x = x[:max_samples]

    # Normalize RMS
    x = normalize(x)

    return fs, x

# -------------------------------------------------
# Load real speech from WAV file
# -------------------------------------------------

wav_path = Path("sounds") / "havard.wav"

fs, speech = load_wav_speech(
    path=wav_path,
    target_fs=8000,
    max_seconds=30,
)

N = len(speech)
T = N / fs
t = np.arange(N) / fs

base_noise = make_engine_like_noise(fs, T, seed=20)

# Make sure noise has exactly same length as speech
base_noise = base_noise[:N]